<a href="https://colab.research.google.com/github/techasit239/DADS7202_PigPicture/blob/main/FreshCheck_Phase2_Exp3_DINOv3_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreshCheck Phase 2 — Exp 3: DINOv3 + Segmentation Head

**Step 1 · Experiment 3: DINOv3 (frozen ViT-S/16) + trainable segmentation head**

> 🔗 **ต้องรัน Phase 2 Foundation ก่อน** — Exp 3 ใช้ Thai test set ที่ Foundation สร้าง

---

## ทำไม Exp 3 ต่างจาก Exp 1+2?

| Exp | Backbone | Train? | Strategy |
|---|---|---|---|
| Exp 1 | SAM 3 | ❌ Zero-shot | Object-centric pretrained — อาจ "หลง" ไปกินบรรจุภัณฑ์ |
| Exp 2 | Florence-2 + SAM 3 | ❌ Zero-shot | 2-stage refinement |
| **Exp 3** | **DINOv3 (frozen)** | ✅ **Train seg head** | **Self-supervised features — material-aware ดีกว่า** |

อาจารย์เขียนใน Comment ว่า: *"DINOv3 จะไม่ค่อย object-centric เท่า SAM3 ถ้าเราเอา DINOv3's feature (frozen) มาต่อ segmentation head ให้มันหน่อย มีแนวโน้มว่าอาจจะ segment เฉพาะ raw meat ออกมาได้ดีกว่า"*

## Architecture

```
Input (H×W×3)
    │
    ▼
DINOv3-ViT-S/16  (frozen, 21M params)
    │
    ▼  patch features (N_patches × 384)
    │
Segmentation Head  (TRAIN)
  - Linear(384, 256) + GELU
  - Linear(256, 1)              ← binary: meat / not meat
    │
    ▼  per-patch logits
    │
Upsample bilinear → (H × W × 1)
    │
    ▼
Binary mask
```

## 5-fold Cross-Validation

Thai test set 150 รูปเล็กมาก ใช้ single split ไม่ได้ — ต้อง **5-fold CV** เพื่อ:
1. **Anti-leakage:** ใช้ piece_id เป็น group → ไม่ให้ก้อนเดียวกันอยู่ทั้ง train + val
2. **Robust metric:** รายงาน mean ± std ของ 5 folds (Proposal commits to this)
3. **ทุกรูปได้ทดสอบ:** ทุกรูปได้เป็น val 1 fold

## เวลา

| ขั้น | งาน | เวลา |
|---|---|---|
| 3.0 | Setup + auth | 1 นาที |
| 3.1 | Load DINOv3 | 2-3 นาที |
| 3.2 | Build dataset + dataloader | 1 นาที |
| 3.3 | Define seg head model | 1 นาที |
| 3.4 | **5-fold CV training** | **~30-45 นาที** |
| 3.5 | Aggregate + visualize | 2 นาที |

**⏱ รวม ~40-60 นาที** (T4 GPU)


## 3.0 — Setup

In [ ]:
# 3.0.1 — Imports + paths
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/FreshCheck'
PHASE2_DIR     = f'{PROJECT_ROOT}/phase2'
THAI_TEST_CSV  = f'{PHASE2_DIR}/thai_test_set.csv'
CONFIGS_DIR    = f'{PROJECT_ROOT}/configs'

EXP3_DIR       = f'{PHASE2_DIR}/exp3_dinov3'
EXP3_MASKS_DIR = f'{EXP3_DIR}/predicted_masks'
EXP3_RESULTS   = f'{EXP3_DIR}/results.json'
os.makedirs(EXP3_DIR, exist_ok=True)
os.makedirs(EXP3_MASKS_DIR, exist_ok=True)

assert os.path.exists(THAI_TEST_CSV), 'รัน Phase 2 Foundation ก่อน'

sys.path.insert(0, CONFIGS_DIR)
from seg_metrics import compute_iou, compute_dice, evaluate_batch
from seg_viz     import show_seg_comparison, plot_iou_distribution
print('[OK] Setup complete')


In [ ]:
# 3.0.2 — HuggingFace login (DINOv3 is gated)
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('[OK] Logged in via Colab Secret')
except Exception:
    login()


## 3.1 — Load DINOv3-ViT-S/16

**Why ViT-S/16 (smallest variant)?**
- 21M params — เล็ก เร็ว memory น้อย เหมาะกับ T4
- Proposal commits to this specific variant
- Patch size 16 → ภาพ 224×224 → 14×14 = 196 patches
- Embedding dim 384

**Frozen:** ไม่เทรน backbone — เทรนแค่ seg head ที่เราเขียน


In [ ]:
# 3.1.1 — Install + load DINOv3
!pip install -q -U transformers
from transformers import AutoModel, AutoImageProcessor

DINOV3_ID = 'facebook/dinov3-vits16-pretrain-lvd1689m'
# (DINOv3-S/16 distilled on LVD-1689M dataset)

print(f'Loading {DINOV3_ID}...')
dinov3_processor = AutoImageProcessor.from_pretrained(DINOV3_ID)
dinov3 = AutoModel.from_pretrained(DINOV3_ID).to(device).eval()

# Freeze backbone — we only train the seg head
for p in dinov3.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in dinov3.parameters())
print(f'[OK] DINOv3 loaded: {n_params/1e6:.1f}M params (frozen)')
print(f'     Embed dim:   {dinov3.config.hidden_size}')
print(f'     Patch size:  {dinov3.config.patch_size}')


## 3.2 — Dataset + Transforms

ภาพแต่ละรูปมีขนาดต่างกัน → resize ให้เท่ากันก่อน (224×224) เพื่อ feed เข้า DINOv3


In [ ]:
# 3.2.1 — Dataset class
IMG_SIZE = 224

img_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

mask_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.NEAREST),
    T.PILToTensor(),  # uint8 0-255
])


class ThaiSegDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.train = train  # for future: add training augmentation

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img  = Image.open(row['image_path']).convert('RGB')
        mask = Image.open(row['mask_path']).convert('L')

        img_t  = img_tf(img)
        mask_t = mask_tf(mask).squeeze(0)
        mask_t = (mask_t > 127).float()  # binarize → 0/1

        return img_t, mask_t, row['filename']


df = pd.read_csv(THAI_TEST_CSV)
print(f'Test set: {len(df)} samples')
print(f'  Unique pieces: {df["piece_id"].nunique()}')
print(f'  Per class: {df["class"].value_counts().to_dict()}')


## 3.3 — Segmentation Head Model

โครงสร้าง:
1. ใส่ภาพเข้า DINOv3 frozen → ได้ patch features (14×14 patches × 384 dim)
2. Linear → 256 → GELU → Linear → 1 (binary logit ต่อ patch)
3. Reshape เป็น (14×14) → bilinear upsample เป็น 224×224
4. Sigmoid → ได้ probability mask


In [ ]:
# 3.3.1 — Seg head model
class DINOv3SegHead(nn.Module):
    """
    DINOv3 (frozen) + simple seg head.
    Output: binary mask logits (B, 1, H, W) — apply sigmoid for probability.
    """
    def __init__(self, dinov3_model, hidden_dim=256, img_size=224, patch_size=16):
        super().__init__()
        self.dinov3 = dinov3_model
        self.embed_dim = dinov3_model.config.hidden_size
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches_per_side = img_size // patch_size  # 14

        # Seg head: per-patch classifier
        self.head = nn.Sequential(
            nn.Linear(self.embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # x: (B, 3, H, W)
        B = x.size(0)

        with torch.no_grad():
            # last_hidden_state: (B, n_register + n_patches + 1[CLS], embed_dim)
            outputs = self.dinov3(pixel_values=x)
            features = outputs.last_hidden_state

        # Strip CLS + register tokens — keep only patch tokens
        # DINOv3 has 1 CLS + 4 register tokens → patches start at index 5
        n_special = 1 + getattr(self.dinov3.config, 'num_register_tokens', 4)
        patch_features = features[:, n_special:, :]  # (B, n_patches, embed_dim)

        # Make sure we have the expected number of patches
        n_patches = self.n_patches_per_side ** 2
        patch_features = patch_features[:, :n_patches, :]

        # Per-patch classification
        logits = self.head(patch_features).squeeze(-1)  # (B, n_patches)

        # Reshape to spatial → (B, 1, h, w)
        h = w = self.n_patches_per_side
        logits = logits.view(B, 1, h, w)

        # Upsample to image resolution
        logits = F.interpolate(logits, size=(self.img_size, self.img_size),
                                mode='bilinear', align_corners=False)
        return logits


model = DINOv3SegHead(dinov3).to(device)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Trainable params: {n_trainable:>10,} ({n_trainable/1e6:.2f}M)  ← seg head only')
print(f'Frozen params:    {n_frozen:>10,} ({n_frozen/1e6:.2f}M)  ← DINOv3 backbone')


## 3.4 — 5-Fold Cross-Validation Training

แต่ละ fold:
1. แบ่ง train (~120 รูป) / val (~30 รูป) โดย group ตาม piece_id
2. Reset model + train head 30 epochs
3. Save best (highest val IoU)
4. Predict + save predicted mask สำหรับ val fold นี้

**Why StratifiedGroupKFold ไม่ใช่ KFold ปกติ:**
- Anti-leakage: ก้อนเนื้อเดียวกันต้องอยู่ split เดียว
- Stratification: balance ทุก class ใน fold

ตอนจบ → ทุก 150 รูปได้ predict 1 ครั้ง (จาก fold ที่มันเป็น val)


In [ ]:
# 3.4.1 — Training loop helpers
def dice_loss(logits, targets, smooth=1.0):
    """Differentiable Dice loss for binary segmentation."""
    probs = torch.sigmoid(logits).flatten(1)
    targets = targets.flatten(1)
    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()


def combo_loss(logits, targets):
    """BCE + Dice — stable for imbalanced segmentation."""
    bce  = F.binary_cross_entropy_with_logits(logits.squeeze(1), targets)
    dice = dice_loss(logits.squeeze(1), targets)
    return 0.5 * bce + 0.5 * dice


def train_one_fold(model, train_loader, val_loader, epochs=30, lr=1e-3):
    """Train one fold. Returns best model state + best metrics."""
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_iou = -1
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        train_losses = []
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = combo_loss(logits, masks)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        # Validate
        model.eval()
        val_ious, val_dices = [], []
        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                logits = model(imgs)
                probs  = torch.sigmoid(logits).squeeze(1)
                preds  = (probs > 0.5).float()
                for p, g in zip(preds, masks):
                    p_np = (p.cpu().numpy() * 255).astype(np.uint8)
                    g_np = (g.cpu().numpy() * 255).astype(np.uint8)
                    val_ious.append(compute_iou(p_np, g_np))
                    val_dices.append(compute_dice(p_np, g_np))

        mean_iou  = float(np.mean(val_ious))
        mean_dice = float(np.mean(val_dices))
        history.append({'epoch': epoch, 'train_loss': float(np.mean(train_losses)),
                         'val_iou': mean_iou, 'val_dice': mean_dice})

        if mean_iou > best_iou:
            best_iou = mean_iou
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state, history, best_iou


In [ ]:
# 3.4.2 — Run 5-fold CV
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

N_FOLDS = 5
EPOCHS = 30
BATCH_SIZE = 8

# Map class to integer for stratification
class_to_int = {c: i for i, c in enumerate(sorted(df['class'].unique()))}
y_strat = df['class'].map(class_to_int).values
groups  = df['piece_id'].values

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

all_pred_masks = {}   # filename → predicted mask (np.uint8)
all_per_sample = []
fold_summary = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, y=y_strat, groups=groups), start=1):
    print(f'\n══════════ FOLD {fold}/{N_FOLDS} ══════════')
    print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

    # Verify anti-leakage
    train_pieces = set(df.iloc[train_idx]['piece_id'])
    val_pieces   = set(df.iloc[val_idx]['piece_id'])
    assert len(train_pieces & val_pieces) == 0, 'LEAK!'

    train_df_f = df.iloc[train_idx].reset_index(drop=True)
    val_df_f   = df.iloc[val_idx].reset_index(drop=True)

    train_ds = ThaiSegDataset(train_df_f, train=True)
    val_ds   = ThaiSegDataset(val_df_f,   train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Fresh model each fold (only head re-init; backbone shared)
    fold_model = DINOv3SegHead(dinov3).to(device)
    best_state, history, best_iou = train_one_fold(fold_model, train_loader, val_loader,
                                                     epochs=EPOCHS, lr=1e-3)
    fold_model.load_state_dict(best_state)

    # Inference on val fold + collect predicted masks
    fold_model.eval()
    val_ious, val_dices = [], []
    with torch.no_grad():
        for imgs, masks, fnames in val_loader:
            imgs = imgs.to(device)
            probs = torch.sigmoid(fold_model(imgs)).squeeze(1).cpu().numpy()
            for i, fname in enumerate(fnames):
                # Resize prediction back to original image size
                orig_row = df[df['filename'] == fname].iloc[0]
                pred_full = Image.fromarray((probs[i] * 255).astype(np.uint8)).resize(
                    (int(orig_row['width']), int(orig_row['height'])), Image.BILINEAR
                )
                pred_arr = (np.array(pred_full) > 127).astype(np.uint8) * 255

                # Save
                Image.fromarray(pred_arr).save(f'{EXP3_MASKS_DIR}/{Path(fname).stem}_pred.png')
                all_pred_masks[fname] = pred_arr

                # Recompute metric at original resolution
                gt_full = np.array(Image.open(orig_row['mask_path']).convert('L'))
                iou  = compute_iou(pred_arr, gt_full)
                dice = compute_dice(pred_arr, gt_full)
                val_ious.append(iou)
                val_dices.append(dice)
                all_per_sample.append({
                    'filename': fname, 'fold': fold,
                    'class': orig_row['class'], 'source': orig_row['source'],
                    'iou': iou, 'dice': dice,
                })

    fold_summary.append({
        'fold': fold,
        'best_val_iou_224':  best_iou,
        'mean_iou_full_res':  float(np.mean(val_ious)),
        'mean_dice_full_res': float(np.mean(val_dices)),
        'n_val': len(val_idx),
    })
    print(f'Fold {fold} done: best val IoU (224px) = {best_iou:.4f} | '
          f'IoU (full res) = {np.mean(val_ious):.4f} | Dice = {np.mean(val_dices):.4f}')

print('\n[OK] 5-fold CV complete')


## 3.5 — Aggregate Across 5 Folds + Visualize

In [ ]:
# 3.5.1 — Aggregate
per_sample_df = pd.DataFrame(all_per_sample)
fold_df = pd.DataFrame(fold_summary)

print(f'5-FOLD CV RESULTS:\n')
print(fold_df.to_string(index=False))

mean_iou_overall  = per_sample_df['iou'].mean()
std_iou_overall   = per_sample_df['iou'].std()
mean_dice_overall = per_sample_df['dice'].mean()
std_dice_overall  = per_sample_df['dice'].std()

print(f'\nOVERALL (all 150 samples across 5 folds):')
print(f'  IoU:  {mean_iou_overall:.4f} ± {std_iou_overall:.4f}')
print(f'  Dice: {mean_dice_overall:.4f} ± {std_dice_overall:.4f}')

print(f'\nFold-level mean (more conservative — accounts for fold variance):')
print(f'  IoU:  {fold_df["mean_iou_full_res"].mean():.4f} ± {fold_df["mean_iou_full_res"].std():.4f}')
print(f'  Dice: {fold_df["mean_dice_full_res"].mean():.4f} ± {fold_df["mean_dice_full_res"].std():.4f}')

print(f'\nPer class:')
for cls in per_sample_df['class'].unique():
    sub = per_sample_df[per_sample_df['class'] == cls]
    print(f'  {cls:12s}: IoU={sub["iou"].mean():.3f}, Dice={sub["dice"].mean():.3f}')

print(f'\nPer source:')
for src in per_sample_df['source'].unique():
    sub = per_sample_df[per_sample_df['source'] == src]
    print(f'  {src:12s}: IoU={sub["iou"].mean():.3f}, Dice={sub["dice"].mean():.3f}')


In [ ]:
# 3.5.2 — Visualize: per-fold + best/median/worst predictions
# Per-fold bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(fold_df['fold'], fold_df['mean_iou_full_res'], color='#1e3a5f', alpha=0.8)
axes[0].set_xlabel('Fold'); axes[0].set_ylabel('Mean IoU')
axes[0].set_title('Per-Fold IoU (full resolution)'); axes[0].grid(alpha=0.3, axis='y')
axes[0].axhline(fold_df['mean_iou_full_res'].mean(), color='#c4583a', linestyle='--',
                 label=f'overall={fold_df["mean_iou_full_res"].mean():.3f}')
axes[0].legend()

axes[1].bar(fold_df['fold'], fold_df['mean_dice_full_res'], color='#4a7c59', alpha=0.8)
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Mean Dice')
axes[1].set_title('Per-Fold Dice (full resolution)'); axes[1].grid(alpha=0.3, axis='y')
axes[1].axhline(fold_df['mean_dice_full_res'].mean(), color='#c4583a', linestyle='--',
                 label=f'overall={fold_df["mean_dice_full_res"].mean():.3f}')
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{EXP3_DIR}/fold_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

# IoU/Dice distribution
results_for_plot = {
    'iou_list':  per_sample_df['iou'].tolist(),
    'dice_list': per_sample_df['dice'].tolist(),
    'mean_iou':  mean_iou_overall,
    'mean_dice': mean_dice_overall,
}
plot_iou_distribution(results_for_plot, figsize=(12, 4))
plt.suptitle('Exp 3: DINOv3 + Seg Head — Distribution', fontsize=12)
plt.tight_layout()
plt.savefig(f'{EXP3_DIR}/iou_dice_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# 3.5.3 — Show best / median / worst predictions
sorted_df = per_sample_df.sort_values('iou').reset_index(drop=True)
for label, pos in [('Worst', 0),
                    ('Median', len(sorted_df) // 2),
                    ('Best', len(sorted_df) - 1)]:
    row = sorted_df.iloc[pos]
    orig = df[df['filename'] == row['filename']].iloc[0]
    pred = all_pred_masks[row['filename']]
    gt   = np.array(Image.open(orig['mask_path']).convert('L'))
    fig = show_seg_comparison(
        orig['image_path'], gt, pred,
        title=f'{label} · fold={row["fold"]} · {row["filename"][:30]} · IoU={row["iou"]:.3f}'
    )
    plt.show()


In [ ]:
# 3.5.4 — Save results
final = {
    'experiment': 'Phase 2 Exp 3 — DINOv3 (frozen) + seg head',
    'model_id': DINOV3_ID,
    'cv_method': 'StratifiedGroupKFold (k=5, group=piece_id, stratify=class)',
    'config': {
        'n_folds': N_FOLDS, 'epochs_per_fold': EPOCHS,
        'batch_size': BATCH_SIZE, 'lr': 1e-3, 'weight_decay': 1e-4,
        'loss': 'BCE + Dice (50/50)',
        'img_size': IMG_SIZE, 'patch_size': 16, 'embed_dim': 384,
        'seed': SEED,
    },
    'fold_summary': fold_summary,
    'aggregate_pooled': {
        'mean_iou':  float(mean_iou_overall),  'std_iou':  float(std_iou_overall),
        'mean_dice': float(mean_dice_overall), 'std_dice': float(std_dice_overall),
        'n_samples': len(per_sample_df),
    },
    'aggregate_fold_level': {
        'mean_iou':  float(fold_df['mean_iou_full_res'].mean()),
        'std_iou':   float(fold_df['mean_iou_full_res'].std()),
        'mean_dice': float(fold_df['mean_dice_full_res'].mean()),
        'std_dice':  float(fold_df['mean_dice_full_res'].std()),
    },
    'per_class': {
        cls: {
            'iou_mean':  float(per_sample_df[per_sample_df['class'] == cls]['iou'].mean()),
            'dice_mean': float(per_sample_df[per_sample_df['class'] == cls]['dice'].mean()),
        } for cls in per_sample_df['class'].unique()
    },
    'per_source': {
        src: {
            'iou_mean':  float(per_sample_df[per_sample_df['source'] == src]['iou'].mean()),
            'dice_mean': float(per_sample_df[per_sample_df['source'] == src]['dice'].mean()),
        } for src in per_sample_df['source'].unique()
    },
    'output_dirs': {'predicted_masks': EXP3_MASKS_DIR},
}

with open(EXP3_RESULTS, 'w') as f:
    json.dump(final, f, indent=2, ensure_ascii=False)
per_sample_df.to_csv(f'{EXP3_DIR}/per_sample_metrics.csv', index=False)
fold_df.to_csv(f'{EXP3_DIR}/fold_summary.csv', index=False)

print(f'[OK] Saved:')
print(f'     {EXP3_RESULTS}')
print(f'     {EXP3_DIR}/per_sample_metrics.csv')
print(f'     {EXP3_DIR}/fold_summary.csv')
print(f'     {EXP3_MASKS_DIR}/ ({len(all_pred_masks)} predicted mask PNG files)')


In [ ]:
# 3.5.5 — Final summary
print('=' * 60)
print('PHASE 2 — Exp 3 (DINOv3 + seg head, 5-fold CV) — DONE')
print('=' * 60)
print(f'\nBackbone: {DINOV3_ID}  (frozen)')
print(f'CV: {N_FOLDS}-fold StratifiedGroupKFold (anti-leakage)')
print(f'\nResults on Thai test set (pooled across 5 folds):')
print(f'  Mean IoU:  {mean_iou_overall:.4f} ± {std_iou_overall:.4f}')
print(f'  Mean Dice: {mean_dice_overall:.4f} ± {std_dice_overall:.4f}')
print(f'\nFold-level summary (more conservative):')
print(f'  IoU:  {fold_df["mean_iou_full_res"].mean():.4f} ± {fold_df["mean_iou_full_res"].std():.4f}')
print(f'  Dice: {fold_df["mean_dice_full_res"].mean():.4f} ± {fold_df["mean_dice_full_res"].std():.4f}')
print(f'\n[OK] All 3 experiments ready for comparison')
print(f'\nNext step: Compare exp1/exp2/exp3/results.json → pick best for Phase 3')
